# Customer Segmentation Using Clustering
## Teresa Ferrill
### October 8, 2026

This notebook examines customer purchasing behavior using transaction data from the UCI Online Retail dataset. The objective is to identify customer groups with similar purchasing characteristics using Recency, Frequency, and Monetary (RFM) analysis and K-Means clustering.

The analysis includes data preparation, exploratory analysis, customer-level feature engineering, clustering, model evaluation, and interpretation of the resulting customer segments.

The original dataset is maintained locally as a ZIP archive and loaded directly into the notebook. Run this notebook from the notebooks directory.

### 1 - Project Overview  
#### Business Problem  
Retail businesses collect transaction data that can provide insight into customer purchasing behavior. However, customers do not necessarily purchase with the same frequency, spend similar amounts, or remain active for the same periods. Identifying these differences can help businesses develop more appropriate marketing and customer engagement strategies.  

#### Research Question  
Can customer purchasing patterns be used to identify meaningful customer segments through RFM analysis and K-Means clustering?

#### Objectives  
1. Examine transaction data to understand customer purchasing behavior.  
2. Identify and address data quality issues that could affect the analysis.
3. Develop customer-level Recency, Frequency, and Monetary (RFM) features.
4. Apply K-Means clustering to identify groups of customers with similar purchasing characteristics.
5. Evaluate cluster quality and interpret the resulting customer segments.
6. Discuss how the identified segments could support business decisions.  

### 2 - Data collection

The dataset for this project is the Online Retail dataset from the UCI Machine Learning Repository. It contains transaction records from a UK-based online retail business between December 2010 and December 2011.  

The dataset includes transaction identifiers, product information, quantities, prices, transaction dates, customer identifiers, and customer countries.  

The original dataset is stored locally as a ZIP archive.  The notebook reads the Excel workbook directly from the archive without modifying the source file.  

Source:
https://archive.ics.uci.edu/dataset/352/online+retail

In [1]:
from pathlib import Path
from zipfile import ZipFile
from io import BytesIO
import pandas as pd

# Check that openpyxl is installed
try:
    import openpyxl
    print(f"openpyxl version {openpyxl.__version__} is available.")
except ImportError:
    raise ImportError(
        "openpyxl is required to read the Excel dataset. "
        "Install it using: pip install openpyxl"
    )

# Notebook is executed from the notebooks directory
DATA_DIR = Path("../data")
ZIP_PATH = DATA_DIR / "online+retail.zip"

if not ZIP_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found: {ZIP_PATH.resolve()}"
    )

# Inspect the archive contents
with ZipFile(ZIP_PATH) as archive:
    print(archive.namelist())


openpyxl version 3.1.5 is available.
['Online Retail.xlsx']


In [2]:
# Load the Excel workbook
with ZipFile(ZIP_PATH) as archive:
    excel_files = [
        name for name in archive.namelist()
        if name.lower().endswith((".xlsx", ".xls"))
        and not Path(name).name.startswith("~$")
    ]

    if len(excel_files) != 1:
        raise ValueError(
            f"Expected one Excel file, found: {excel_files}"
        )

    with archive.open(excel_files[0]) as file:
        excel_bytes = BytesIO(file.read())

    df = pd.read_excel(excel_bytes)

print(f"Dataset shape: {df.shape}")
display(df.head())


Dataset shape: (541909, 8)


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


### 3 - Data Understanding 
Structure  
data types  
missing values  

In [3]:
# Examine dataset structure and data types
print("Dataset Information")
df.info()

# Review missing values
print("\nMissing Values")
display(
    pd.DataFrame({
        "Missing Count": df.isna().sum(),
        "Missing Percentage": (
            df.isna().mean() * 100
        ).round(2)
    })
)

# Identify duplicate records
print(f"\nDuplicate records: {df.duplicated().sum():,}")

# Review transaction date range
print(f"\nEarliest transaction: {df['InvoiceDate'].min()}")
print(f"Latest transaction: {df['InvoiceDate'].max()}")

# Count unique customers
print(
    f"\nUnique customers: "
    f"{df['CustomerID'].nunique():,}"
)


Dataset Information
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    541909 non-null  object        
 1   StockCode    541909 non-null  object        
 2   Description  540455 non-null  object        
 3   Quantity     541909 non-null  int64         
 4   InvoiceDate  541909 non-null  datetime64[ns]
 5   UnitPrice    541909 non-null  float64       
 6   CustomerID   406829 non-null  float64       
 7   Country      541909 non-null  object        
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 33.1+ MB

Missing Values


,Missing Count,Missing Percentage
InvoiceNo,0,0.00
StockCode,0,0.00
Description,1454,0.27
Quantity,0,0.00
InvoiceDate,0,0.00
UnitPrice,0,0.00
CustomerID,135080,24.93
Country,0,0.00



Duplicate records: 5,268

Earliest transaction: 2010-12-01 08:26:00
Latest transaction: 2011-12-09 12:50:00

Unique customers: 4,372


-------------------------------------------------------------------------------------------------------
#### Additional Transaction Quality Assessment

In [4]:
# Examine transaction characteristics before cleaning

print("Quantity Summary")
display(df["Quantity"].describe())

print("\nUnit Price Summary")
display(df["UnitPrice"].describe())

# Identify nonpositive quantities and prices
print(
    f"\nQuantity <= 0: "
    f"{(df['Quantity'] <= 0).sum():,}"
)
print(
    f"UnitPrice <= 0: "
    f"{(df['UnitPrice'] <= 0).sum():,}"
)

# Identify cancellation invoices
cancellations = (
    df["InvoiceNo"]
    .astype(str)
    .str.upper()
    .str.startswith("C")
)

print(
    f"Cancellation transaction rows: "
    f"{cancellations.sum():,}"
)

print(
    f"Unique cancellation invoices: "
    f"{df.loc[cancellations, 'InvoiceNo'].nunique():,}"
)

# Examine transaction distribution by country
print("\nTop 10 Countries")
display(df["Country"].value_counts().head(10))


Quantity Summary


count    541909.000000
mean          9.552250
std         218.081158
min      -80995.000000
25%           1.000000
50%           3.000000
75%          10.000000
max       80995.000000
Name: Quantity, dtype: float64


Unit Price Summary


count    541909.000000
mean          4.611114
std          96.759853
min      -11062.060000
25%           1.250000
50%           2.080000
75%           4.130000
max       38970.000000
Name: UnitPrice, dtype: float64


Quantity <= 0: 10,624
UnitPrice <= 0: 2,517
Cancellation transaction rows: 9,288
Unique cancellation invoices: 3,836

Top 10 Countries


Country
United Kingdom    495478
Germany             9495
France              8557
EIRE                8196
Spain               2533
Netherlands         2371
Belgium             2069
Switzerland         2002
Portugal            1519
Australia           1259
Name: count, dtype: int64

-------------------------------------------------------------------------------------------------------
#### Initial Data Quality Findings

The dataset contains 541,909 transaction records across eight columns, representing 4,372 unique customers. Transaction dates range from December 1, 2010, through December 9, 2011.  

The initial review identified 135,080 records (24.93%) without a customer identifier, 1,454 records (0.27%) with missing product descriptions, and 5,268 exact duplicate rows.  

Missing customer identifiers are particularly important because the planned RFM analysis requires transactions to be associated with individual customers. Records without a customer identifier will therefore be excluded from customer-level analysis.  

The additional review identified 10,624 records with nonpositive quantities, 2,517 records with nonpositive unit prices, and 9,288 rows associated with cancellation invoices. The data also contains unusually large positive and negative transaction values that require consideration during cleaning.  

Most transaction records originate from the United Kingdom. The analysis will initially retain customers from all countries while excluding records that cannot support the intended customer-level purchasing analysis.

-------------------------------------------------------------------------------------------------------
### 4 - Data Cleaning
Cleaning process to preserve the original dataset:
> 1. Missing Customer ID - Exclude  
> 2. Cancellation invoice (c previx) - Exclude from puchase-only RFM
> 3. Quantity <= 0 - Exclude
> 4. UnitPrice <= 0 - Exclude
> 5. Exact duplicate rows - Remove (documenting the assumption)
> 6. Missing Description - Retain if other require fields are valid
> 7. Country - Retain all initially

In [5]:
# Data Cleaning
# Preserve the original dataset
df_clean = df.copy()
initial_rows = len(df_clean)

# Track cleaning results
cleaning_summary = []

def record_step(step, before, after):
    cleaning_summary.append({
        "Step": step,
        "Rows Removed": before - after,
        "Rows Remaining": after
    })

# 1. Remove missing customer identifiers
before = len(df_clean)
df_clean = df_clean.dropna(subset=["CustomerID"])
record_step("Missing CustomerID", before, len(df_clean))

# 2. Remove cancellation invoices
before = len(df_clean)
df_clean = df_clean[
    ~df_clean["InvoiceNo"]
    .astype(str)
    .str.upper()
    .str.startswith("C")
].copy()
record_step("Cancellation invoices", before, len(df_clean))

# 3. Retain positive quantities
before = len(df_clean)
df_clean = df_clean[df_clean["Quantity"] > 0].copy()
record_step("Nonpositive quantity", before, len(df_clean))

# 4. Retain positive unit prices
before = len(df_clean)
df_clean = df_clean[df_clean["UnitPrice"] > 0].copy()
record_step("Nonpositive unit price", before, len(df_clean))

# 5. Remove exact duplicate rows
before = len(df_clean)
df_clean = df_clean.drop_duplicates().copy()
record_step("Exact duplicates", before, len(df_clean))

# Display cleaning summary
display(pd.DataFrame(cleaning_summary))

print(f"Original rows: {initial_rows:,}")
print(f"Cleaned rows: {len(df_clean):,}")
print(f"Total removed: {initial_rows - len(df_clean):,}")
print(
    f"Records retained: "
    f"{len(df_clean) / initial_rows * 100:.2f}%"
)


,Step,Rows Removed,Rows Remaining
0,Missing CustomerID,135080,406829
1,Cancellation invoices,8905,397924
2,Nonpositive quantity,0,397924
3,Nonpositive unit price,40,397884
4,Exact duplicates,5192,392692


Original rows: 541,909
Cleaned rows: 392,692
Total removed: 149,217
Records retained: 72.46%


In [6]:
# Validate cleaned dataset

print("Remaining missing CustomerID:",
      df_clean["CustomerID"].isna().sum())

print("Remaining nonpositive quantities:",
      (df_clean["Quantity"] <= 0).sum())

print("Remaining nonpositive prices:",
      (df_clean["UnitPrice"] <= 0).sum())

print("Remaining duplicates:",
      df_clean.duplicated().sum())

print("Unique customers:",
      df_clean["CustomerID"].nunique())

# Verify all cleaning conditions
assert df_clean["CustomerID"].notna().all()
assert (df_clean["Quantity"] > 0).all()
assert (df_clean["UnitPrice"] > 0).all()
assert not df_clean.duplicated().any()
assert not df_clean["InvoiceNo"].astype(str).str.upper().str.startswith("C").any()


Remaining missing CustomerID: 0
Remaining nonpositive quantities: 0
Remaining nonpositive prices: 0
Remaining duplicates: 0
Unique customers: 4338


### 5 - Exploratory Data Analysis (EDA)  
Recency  
Frequency  
Monetary values  

### 6 - Feature Engineering
Recency  
Frequency  
Monetary values

### 7 - Data Processing
Transformations and scaling

### 8 - K-Means Clustering
Evaluate different numbers of clusters

### 9 - Cluster Evaluation
Silhouette scores and cluster characteristics  

### 10 - Business Interpretation 
Customer profiles and recommendations

### 11 - Conclusions
Findings  
Limitations  
Future Improvements